In [12]:
import pandas as pd

In [13]:
occ_summary = pd.read_csv('../outputs/occ_summary.csv')
occ_summary['NCO_3digit'] = (
    pd.to_numeric(occ_summary['NCO_3digit'], errors='coerce')
      .astype('Int64')
      .astype('string')
      .str.zfill(3)
)

In [14]:
ilo_df = pd.read_csv('../data/ilo_wp140_table_a1.csv')

print("Shape:", ilo_df.shape)
print("Columns:", ilo_df.columns.tolist())
ilo_df.head(10)

Shape: (3265, 21)
Columns: ['label4d', 'label1d', 'ISCO_08', 'Title', 'taskID', 'Task_ISCO', 'score_2023', 'Weaviate Status', 'predicted_score_2025_gpt4o', 'prediction_justification_gpt4o', 'weaviate_status_gemini', 'predicted_score_2025_gemini', 'prediction_justification_gemini', 'score_2025', 'source', 'mean_score_2023', 'mean_score_2025', 'SD_2023', 'SD_2025', 'potential25', 'potential23']


,label4d,label1d,ISCO_08,Title,taskID,Task_ISCO,score_2023,Weaviate Status,predicted_score_2025_gpt4o,prediction_justification_gpt4o,...,predicted_score_2025_gemini,prediction_justification_gemini,score_2025,source,mean_score_2023,mean_score_2025,SD_2023,SD_2025,potential25,potential23
0,1111 - Legislators,1 - Managers,1111,Legislators,1,Presiding over or participating in the proceed...,0.1,success,0.330,The task of presiding over or participating in...,...,0.35,The task of presiding over legislative proceed...,0.3400,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
1,1111 - Legislators,1 - Managers,1111,Legislators,2,"Determining, formulating and directing policie...",0.3,success,0.365,"The task of determining, formulating, and dire...",...,0.35,"The task of ""Determining, formulating and dire...",0.3575,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
2,1111 - Legislators,1 - Managers,1111,Legislators,3,"Making, ratifying, amending or repealing laws,...",0.2,success,0.330,"The task of making, ratifying, amending, or re...",...,0.25,"The task of making, ratifying, amending, or re...",0.2900,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
3,1111 - Legislators,1 - Managers,1111,Legislators,4,Serving on government administrative boards or...,0.2,success,0.220,"The task of ""Serving on government administrat...",...,0.28,The task of serving on government administrati...,0.2500,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
4,1111 - Legislators,1 - Managers,1111,Legislators,5,Investigating matters of concern to the public...,0.3,success,0.340,"The task of ""Investigating matters of concern ...",...,0.35,"The task of ""Investigating matters of concern ...",0.3450,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
5,1111 - Legislators,1 - Managers,1111,Legislators,6,Attending community functions and meetings to ...,0.2,success,0.320,The task of attending community functions and ...,...,0.35,The task of attending community functions and ...,0.3350,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
6,1111 - Legislators,1 - Managers,1111,Legislators,7,Negotiating with other legislators and represe...,0.2,success,0.275,The task of negotiating with legislators and r...,...,0.25,The task of negotiating with legislators and i...,0.2625,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
7,1111 - Legislators,1 - Managers,1111,Legislators,8,"As members of the government, directing senior...",0.3,success,0.300,The task of directing senior administrators an...,...,0.35,The task of directing senior administrators an...,0.3250,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
8,1112 - Senior Government Officials,1 - Managers,1112,Senior Government Officials,1,"Advising national, state, regional or local go...",0.3,success,0.350,"Advising national, state, regional, or local g...",...,0.35,The task of advising governments and legislato...,0.3500,predicted,0.32,0.38,0.16,0.08,Minimal Exposure,Not affected
9,1112 - Senior Government Officials,1 - Managers,1112,Senior Government Officials,2,Advising on the preparation of government budg...,0.3,success,0.420,The task of advising on the preparation of gov...,...,0.45,The task of advising on the preparation of gov...,0.4350,predicted,0.32,0.38,0.16,0.08,Minimal Exposure,Not affected


In [15]:
ilo_df = ilo_df.rename(columns={
    'ISCO_08': 'ISCO_4digit',   # <- replace with actual column name
    'mean_score_2025': 'Mean_Score',             # <- replace with actual column name
    'SD_2025': 'SD_Score'
})

# Force ISCO code to text with leading zeros preserved
ilo_df['ISCO_4digit'] = (
    pd.to_numeric(ilo_df['ISCO_4digit'], errors='coerce')
      .astype('Int64')
      .astype('string')
      .str.zfill(4)
)

ilo_df[['ISCO_4digit', 'Mean_Score', 'SD_Score']].head(10)

,ISCO_4digit,Mean_Score,SD_Score
0,1111,0.31,0.04
1,1111,0.31,0.04
2,1111,0.31,0.04
3,1111,0.31,0.04
4,1111,0.31,0.04
5,1111,0.31,0.04
6,1111,0.31,0.04
7,1111,0.31,0.04
8,1112,0.38,0.08
9,1112,0.38,0.08


In [16]:
#Aggregate 4-digit scores up to 3-digit level

ilo_df['ISCO_3digit'] = ilo_df['ISCO_4digit'].str[:3]

# Scores are repeated across task rows; retain one row per occupation first.
occupation_scores = ilo_df.drop_duplicates(subset='ISCO_4digit')

crosswalk = occupation_scores.groupby('ISCO_3digit').agg(
    Mean_Exposure_Score=('Mean_Score', 'mean'),
    N_Constituent_Occupations=('ISCO_4digit', 'nunique')
).reset_index()

print("Number of 3-digit groups with ILO data:", crosswalk.shape[0])
crosswalk.head(15)

Number of 3-digit groups with ILO data: 127


,ISCO_3digit,Mean_Exposure_Score,N_Constituent_Occupations
0,111,0.322500,4
1,112,0.380000,1
2,121,0.377500,4
3,122,0.400000,3
4,131,0.245000,2
5,132,0.362500,4
6,133,0.440000,1
7,134,0.368571,7
8,141,0.365000,2
9,142,0.440000,1


In [17]:
#Approximate exposure gradient
def assign_gradient(m):
    if m >= 0.6:
        return 'Gradient 4'
    elif m >= 0.5:
        return 'Gradient 3'
    elif m >= 0.4:
        return 'Gradient 2'
    elif m >= 0.3:
        return 'Gradient 1 / Minimal'
    else:
        return 'Not Exposed'

crosswalk['Approx_Gradient'] = crosswalk['Mean_Exposure_Score'].apply(assign_gradient)
crosswalk.head(15)

,ISCO_3digit,Mean_Exposure_Score,N_Constituent_Occupations,Approx_Gradient
0,111,0.322500,4,Gradient 1 / Minimal
1,112,0.380000,1,Gradient 1 / Minimal
2,121,0.377500,4,Gradient 1 / Minimal
3,122,0.400000,3,Gradient 1 / Minimal
4,131,0.245000,2,Not Exposed
5,132,0.362500,4,Gradient 1 / Minimal
6,133,0.440000,1,Gradient 2
7,134,0.368571,7,Gradient 1 / Minimal
8,141,0.365000,2,Gradient 1 / Minimal
9,142,0.440000,1,Gradient 2


In [18]:
#Merge the crosswalk and PLFS occupation list
master_crosswalk = occ_summary.merge(
    crosswalk,
    left_on='NCO_3digit',
    right_on='ISCO_3digit',
    how='left'
)

master_crosswalk['Match_Status'] = master_crosswalk['Mean_Exposure_Score'].apply(
    lambda x: 'Matched' if pd.notna(x) else 'Unmatched'
)

master_crosswalk.head(20)

,NCO_3digit,Unweighted_Count,Weighted_Employment,ISCO_3digit,Mean_Exposure_Score,N_Constituent_Occupations,Approx_Gradient,Match_Status
0,522,8912,796098362,522,0.390000,3,Gradient 1 / Minimal,Matched
1,911,5646,667867631,911,0.130000,2,Not Exposed,Matched
2,932,5882,628522022,932,0.160000,2,Not Exposed,Matched
3,541,5593,503797321,541,0.172000,5,Not Exposed,Matched
4,234,6517,468850741,234,0.235000,2,Not Exposed,Matched
5,832,4061,387859324,832,0.265000,2,Not Exposed,Matched
6,833,3792,386215942,833,0.210000,2,Not Exposed,Matched
7,233,4714,340507990,233,0.300000,1,Gradient 1 / Minimal,Matched
8,512,3023,338313152,512,0.180000,1,Not Exposed,Matched
9,411,4420,327799527,411,0.600000,1,Gradient 4,Matched


In [19]:
print(master_crosswalk['Match_Status'].value_counts())

Match_Status
Matched    123
Name: count, dtype: int64


In [20]:
#Calculate coverage
matched = master_crosswalk[master_crosswalk['Match_Status'] == 'Matched']
unmatched = master_crosswalk[master_crosswalk['Match_Status'] == 'Unmatched']

# Unweighted coverage: share of unique occupation CODES matched
unweighted_coverage = (matched.shape[0] / master_crosswalk.shape[0]) * 100

# Weighted coverage: share of the actual person-weighted SAMPLE matched
# (this is the number that matters most — it reflects real employment share, not just code count)
total_weighted = master_crosswalk['Weighted_Employment'].sum()
matched_weighted = matched['Weighted_Employment'].sum()
weighted_coverage = (matched_weighted / total_weighted) * 100

print(f"Unweighted coverage (share of occupation codes matched): {unweighted_coverage:.2f}%")
print(f"Weighted coverage (share of regular-salaried sample matched): {weighted_coverage:.2f}%")

Unweighted coverage (share of occupation codes matched): 100.00%
Weighted coverage (share of regular-salaried sample matched): 100.00%


In [21]:
master_crosswalk.to_csv('../outputs/master_crosswalk.csv', index=False)
unmatched.to_csv('../outputs/unmatched_occupations.csv', index=False)

coverage_summary = pd.DataFrame({
    'Metric': ['Unweighted Coverage (%)', 'Weighted Coverage (%)'],
    'Value': [round(unweighted_coverage, 2), round(weighted_coverage, 2)]
})
coverage_summary.to_csv('../outputs/coverage_summary.csv', index=False)

print("All crosswalk outputs saved.")
print(coverage_summary)

All crosswalk outputs saved.
                    Metric  Value
0  Unweighted Coverage (%)  100.0
1    Weighted Coverage (%)  100.0
